In [1]:
#Example 3
from problems_generator import compressible_opt_pb as co_pb
from functools import partial
from troma import CombinatorialProblem, ConstraintSketchMap, matching_pursuit
from troma import get_optimizer

number_spins = 12
rules_reward = {(0, 0, 0, 0): 18, (1, 1, 0, 1): 5, (1, 1, 1, 1): 18}

objective = partial(co_pb.estimate_cost, rules_rewards=rules_reward)

In [ ]:
# 1. Create problem
problem = CombinatorialProblem(objective, problem_size=5, problem_dimension=2)

# 2. Sample
problem.sampling(n_samples=800)

# 3. Create sketch map and sketch
sketch_map = ConstraintSketchMap(sketch_length=5, interaction_size=4, constraints="nearest_neighbors")
problem_sketch = problem.sketching(sketch_map)

# 4. Run matching pursuit
result = matching_pursuit(problem_sketch, iteration_number=5, optimizer=get_optimizer("spin_chain_nn_max"))

# 5. Get results
print(result.positions)      # Selected line positions
# print(result.values)         # Coefficients
# print(result.n_lines)        # Number of selected lines

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31,

AttributeError: 'CombinatorialProblemSketch' object has no attribute 'sketch'

# Embbedings

In [3]:
import numpy as np
from troma.core import data_structure as ds
from troma import spectrum_embedding
from matplotlib import pyplot as plt

number_spins = 12
rules_reward = {(0, 0, 0, 0): 18, (1, 1, 0, 1): 5, (1, 1, 1, 1): 18}

full_spectrum = co_pb.compute_full_spectrum(number_spins, rules_reward)
print(np.where(full_spectrum == np.max(full_spectrum)))
spectrum_pos = []
spectrum_val = []
spectrum_bin = []
for i,j in enumerate(full_spectrum):
    spectrum_pos.append(i)
    spectrum_val.append(j)
    spectrum_bin.append( ds.integer_to_dit_string(i, 12) )

# plt.bar(spectrum_pos, spectrum_val)
# plt.show()

(array([   0, 4095]),)


In [4]:
emb_spectrum = spectrum_embedding(spectrum_bin, additional_dits=[0], dimension_mapping=None, additional_dits_val=1)
emb_pos = [ds.dit_string_to_integer(s, dit_dimension=2) for s in emb_spectrum]
full_emb_spectrum = np.zeros((2**13,))
for pos, val in zip(emb_pos, spectrum_val):
    full_emb_spectrum[pos] = val

print(np.where(full_emb_spectrum == np.max(full_emb_spectrum)))
# plt.bar(range(len(full_emb_spectrum)), full_emb_spectrum)
# plt.show()

(array([4096, 8191]),)


In [5]:
def ev_conf(conf):
    return full_emb_spectrum[ds.dit_string_to_integer(conf, dit_dimension=2)]

In [7]:
#1 create problem
problem = CombinatorialProblem(ev_conf, problem_size=13)

#2 Restrict problem
restricted_problem = problem.restrict(dit_restrictions=np.arange(1,13), dit_value_restrictions=None, additional_dits_val=1)

#3 Sample
restricted_problem.sampling(n_samples=400)

#4 Auto sketch
problem_sketch = restricted_problem.sketching("nearest_neighbors", interaction_size=4)

#5 Run matching pursuit
result = matching_pursuit(problem_sketch, iteration_number=5, optimizer=get_optimizer("spin_chain_nn_max"))

print(result.positions)

[8191 4096 7224 6382 7637]
